# T10 — One-click reproduce of the eval tables (Colab)

Reproduces the headline results of the Multimodal-RAG study (H1–H3, faithfulness, QA-accuracy) from the
**reproducibility pack** `data/processed/repro_pack.zip` (built locally with `scripts/pack_repro.py`).

The pack ships the *prebuilt* indexes — the FAISS text index (BGE-M3 over 1226 chunks) and the CLIP image
index (252 visual chunks) — so retrieval/fusion reproduce **deterministically and without an API key**.
Re-running ingest is avoided on purpose: it costs ~250 Gemini caption calls and is nondeterministic.

| Step | Script | Needs `GEMINI_API_KEY`? | Hypothesis |
|---|---|---|---|
| Retrieval eval | `scripts/evaluate.py` | no | H1 / H2 |
| Cross-modal fusion eval | `scripts/evaluate_fusion.py` | no | H3 |
| Faithfulness / citation | `scripts/faithfulness.py` | **yes** | grounding |
| QA-accuracy (LLM-judge) | `scripts/qa_accuracy.py` | **yes** | answer correctness |

**Workflow:** set runtime to GPU (Runtime → Change runtime type → T4 — speeds the BGE-M3 re-embed in the
retrieval eval), run the cells top to bottom, and upload `repro_pack.zip` when prompted. All pinned model
IDs / seeds / config / package versions are logged in `REPRO_MANIFEST.json` inside the pack.

In [ ]:
# 1. Eval dependencies only (OCR/Paddle + Gradio are not needed to reproduce the tables).
!pip -q install "sentence-transformers>=3.0" "faiss-cpu>=1.8" rank-bm25 "google-genai>=1.0" \
    pyyaml python-dotenv "numpy>=1.26" Pillow pymupdf tqdm
import torch
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# 2. Upload data/processed/repro_pack.zip (from scripts/pack_repro.py) and unzip into a clean root.
import json, zipfile, pathlib, os
from google.colab import files

uploaded = files.upload()  # pick repro_pack.zip
zip_name = next(iter(uploaded))
ROOT = pathlib.Path("/content/mmrag_repro")
with zipfile.ZipFile(zip_name) as z:
    z.extractall(ROOT)
os.chdir(ROOT)  # scripts read gold via paths relative to cwd; config resolves the rest by file location

manifest = json.loads((ROOT / "REPRO_MANIFEST.json").read_text(encoding="utf-8"))
print("git commit :", manifest["git_commit"], "| built", manifest["created_utc"])
print("models     :", json.dumps(manifest["models"], indent=0))
print("seeds      :", manifest["seeds"])
d = manifest["dataset"]
print(f"dataset    : {d['decks']} decks / {d['chunks']} chunks "
      f"({d['text_chunks']} text + {d['figure_chunks']} fig) / "
      f"{d['gold_qa_total']} gold QA ({d['gold_qa_figure']} fig + {d['gold_qa_text']} text)")
print("image index:", d["image_index_visual_chunks"], "visual chunks,", d["image_index_dim"], "-d")

In [ ]:
# 3. Reproduce the RETRIEVAL eval (H1/H2): BM25 vs dense BGE-M3, text-only vs +captions.
#    Re-embeds the 1226 text+caption chunks with BGE-M3 (fast on GPU) and scores the 33 gold Qs.
#    Key-free. The faiss+torch guards are harmless on Linux but kept for parity with local runs.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
!python scripts/evaluate.py --gold data/eval/slides_qa.json

In [ ]:
# 4. Reproduce the CROSS-MODAL FUSION eval (H3): describe-then-embed (text) vs embed-the-image
#    (CLIP) vs RRF / z-linear / fixed-blend fusion, with the α-sensitivity curve. Loads the prebuilt
#    FAISS + CLIP indexes from the pack (no re-embed of the text side). Key-free.
!python scripts/evaluate_fusion.py --gold data/eval/slides_qa.json

## Optional: live-answer evals (need `GEMINI_API_KEY`)

Faithfulness/citation and QA-accuracy generate grounded answers with Gemini, so they need a key. Skip
the next three cells to reproduce only the key-free retrieval/fusion tables above.

In [ ]:
# 5. (optional) Provide the key. Prefer Colab Secrets (key icon) named GEMINI_API_KEY; falls back to a prompt.
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("GEMINI_API_KEY loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY (leave blank to skip live evals): ")
    print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
# 6. (optional) Faithfulness / citation-grounding over the 33 gold Qs (live grounded answers).
if os.environ.get("GEMINI_API_KEY"):
    !python scripts/faithfulness.py --gold data/eval/slides_qa.json
else:
    print("No GEMINI_API_KEY — skipping faithfulness eval.")

In [ ]:
# 7. (optional) QA-accuracy via LLM-judge (gemini-2.5-flash judging gemini-2.5-flash-lite answers).
if os.environ.get("GEMINI_API_KEY"):
    !python scripts/qa_accuracy.py --gold data/eval/slides_qa.json
else:
    print("No GEMINI_API_KEY — skipping QA-accuracy eval.")

## (Reference) GPU rebuild of the CLIP image index

The image index in the pack was built on the T6 path. To rebuild it from scratch on GPU — the genuinely
GPU-heavy step — use the separate notebook **`build_image_index.ipynb`** with the slide-PNG payload
(`scripts/pack_repro.py --with-payload`, or `scripts/build_image_index.py --pack`). It writes
`image_index/` in the `ImageIndex.save` format, which drops into `data/processed/`.